## Posterior predictive checks for continuous variables

### Comparing marginal distributions

In [ ]:
import arviz as az
import xarray as xr

az.style.use("arviz-variat")

In [ ]:
dt = az.load_arviz_data("radon")

In [ ]:
pc = az.plot_dist(
    dt,
    group="posterior_predictive",
    sample_dims="obs_id",
    aes={"overlay": ["chain", "draw"]},
    cols=[],
    visuals={"credible_interval": False, "point_estimate": False, "point_estimate_text": False},
)
az.plot_dist(
    dt,
    group="observed_data",
    sample_dims="obs_id",
    visuals={"dist": {"color": "black"}, "credible_interval": False, "point_estimate": False, "point_estimate_text": False},
    plot_collection=pc,
);

In [ ]:
dt1 = xr.load_datatree("data/base_good.zarr", engine="zarr")
dt_under = xr.load_datatree("data/base_underdispersed.zarr", engine="zarr")
dt_over = xr.load_datatree("data/base_overdispersed.zarr", engine="zarr")
dt_biased = xr.load_datatree("data/base_biased.zarr", engine="zarr")
combined_dt = xr.concat((dt1, dt_under, dt_over, dt_biased), dim=xr.DataArray(["Proper fit", "Underdispersed", "Overdispersed", "Biased"], dims=["model"]))

In [ ]:
pc = az.plot_ppc_dist(
    combined_dt,
    cols=["model"],
    visuals={"observed_dist": {"color": "C0"}}
);

In [ ]:
az.combine_plots(
    combined_dt,
    [
        (az.plot_ppc_dist, {"kind": "kde"}),
        (az.plot_ppc_dist, {"kind": "ecdf"}),
        (az.plot_ppc_dist, {"kind": "dot"}),
        (az.plot_ppc_dist, {"kind": "hist"}),
    ],
    group="posterior_predictive",
    expand="row",
    cols=["model"],
);

In [ ]:
az.combine_plots(
    dt_biased,
    [
        (az.plot_ppc_tstat, {"t_stat": "std"}),
        (az.plot_ppc_tstat, {"t_stat": "mad"}),
        (az.plot_ppc_tstat, {"t_stat": "iqr"})
    ],
    group="posterior_predictive",
    expand="row"
);

In [ ]:
def mode(x):
    return az.mode(x, dim="item")
    
az.combine_plots(
    dt_biased,
    [
        (az.plot_ppc_tstat, {"t_stat": "mean"}),
        (az.plot_ppc_tstat, {"t_stat": "median"}),
        (az.plot_ppc_tstat, {"t_stat": mode})
    ],
    group="posterior_predictive",
    expand="row"
);

### Probability Integral Transform

If we loop over all observations and compute

$$
p(\tilde{y}_i \leq y_i | y) \approx \sum_s I(\tilde{y}_i^s < y_i) / S
$$

Where $s$ represents each realization of our posterior predictive samples, and $i$ indicates the observation index at which the comparision between the respective observations $y$ and posterior predictive samples $\hat{y}$.

In [ ]:
az.plot_dist((dt1.posterior_predictive <= dt1.observed_data).dataset.mean(("chain", "draw")), sample_dims="item");

In [ ]:
az.plot_dist((dt_over.posterior_predictive <= dt_over.observed_data).dataset.mean(("chain", "draw")), sample_dims="item");

In [ ]:
az.plot_ppc_pit(dt_under);

## Categories or hierarchies within continuous data

In [ ]:
az.plot_ppc_dist(dt1, cols=["category"]);

The goal is to have plots adapt to the facetting strategy defined by the user and perform the relevant computations automatically, but not all functions support this yet.
In many cases, we can "promote" those subsets based on the categories or hierarchies to their own variables. Many more plots support multiple variables as input and automatically facet along the different variables whereas the automatic groupby based on the defined facetting strategy is still in progress.

In [ ]:
def explode_category_to_var(ds):
    if ds:
        return az.explode_dataset_dims(ds.set_xindex("category"), "item", dim_to_idx={"item": "category"})
    return ds

dt1_bis = dt1.map_over_datasets(explode_category_to_var)
dt1_bis

In [ ]:
az.plot_ppc_pit(dt1_bis);

## Playground time!

Are there mismatches between the observations and the posterior predictive in these two example datasets?
If so how would you describe them? What visualization helps you see them more clearly?

In [ ]:
dt_ex1 = xr.load_datatree("data/base_ex1.zarr", engine="zarr")
dt_ex2 = xr.load_datatree("data/base_ex2.zarr", engine="zarr")
# you can also use example data "radon" loaded in variable `dt` to familiarize with the plots introduced so far

## See more
[EABM chapter on posterior predictive checks for continuous data](https://arviz-devs.github.io/EABM/Chapters/Prior_posterior_predictive_checks.html#posterior-predictive-checks)